# LR

In [1]:
!pip install scikit-learn scipy joblib

Looking in indexes: http://mirrors.aliyun.com/pypi/simple


In [2]:
import pandas as pd
import numpy as np
import os

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

In [3]:
DATA_PATH = "FeatureA_Repeated"
OUTPUT_PATH = "LR_FeatureA_Results"

os.makedirs(OUTPUT_PATH, exist_ok=True)

N_REPEATS = 10

C_VALUES = [0.1, 1, 10]

N_INNER_REPEATS = 3
VALID_RATIO = 0.2
BASE_SEED = 42

In [4]:
def get_feature_cols(df):
    return [
        col for col in df.columns
        if col not in ["userId", "movieId", "label"]
    ]

In [5]:
def stratified_split_from_scratch(df, label_col, test_ratio=0.2, random_seed=42):
    rng = np.random.default_rng(random_seed)
    train_indices = []
    test_indices = []
    for label_value in df[label_col].unique():
        label_indices = df[df[label_col] == label_value].index.to_numpy()
        rng.shuffle(label_indices)

        test_size = int(len(label_indices) * test_ratio)

        test_indices.extend(label_indices[:test_size])
        train_indices.extend(label_indices[test_size:])

    train_df = df.loc[train_indices].sample(
        frac=1,
        random_state=random_seed
    ).reset_index(drop=True)

    test_df = df.loc[test_indices].sample(
        frac=1,
        random_state=random_seed
    ).reset_index(drop=True)

    return train_df, test_df

In [6]:
def compute_basic_metrics_from_scratch(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    accuracy = (tp + tn) / len(y_true)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0

    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn
    }

In [7]:
def compute_auc_from_scratch(y_true, y_prob):
    y_true = np.array(y_true)
    y_prob = np.array(y_prob)

    sorted_indices = np.argsort(-y_prob)
    y_true_sorted = y_true[sorted_indices]

    pos_count = np.sum(y_true == 1)
    neg_count = np.sum(y_true == 0)

    if pos_count == 0 or neg_count == 0:
        return 0

    tp = 0
    fp = 0

    tpr_list = [0]
    fpr_list = [0]

    for label in y_true_sorted:
        if label == 1:
            tp += 1
        else:
            fp += 1

        tpr_list.append(tp / pos_count)
        fpr_list.append(fp / neg_count)

    auc = 0

    for i in range(1, len(tpr_list)):
        auc += (
            (fpr_list[i] - fpr_list[i - 1])
            *
            (tpr_list[i] + tpr_list[i - 1])
            / 2
        )

    return auc

In [8]:
def train_and_evaluate_lr(train_df, test_df, C_value):
    feature_cols = get_feature_cols(train_df)

    X_train = train_df[feature_cols]
    y_train = train_df["label"]

    X_test = test_df[feature_cols]
    y_test = test_df["label"]

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = LogisticRegression(
        C=C_value,
        penalty="l2",
        solver="liblinear",
        max_iter=1000,
        random_state=42
    )

    model.fit(X_train_scaled, y_train)

    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]

    metrics = compute_basic_metrics_from_scratch(y_test, y_pred)
    metrics["auc"] = compute_auc_from_scratch(y_test, y_prob)

    return metrics

In [9]:
def tune_lr_C_from_scratch(train_df, C_values, n_inner_repeats=3, valid_ratio=0.2, base_seed=100):
    tuning_records = []

    for C_value in C_values:
        inner_f1_scores = []

        for inner_id in range(n_inner_repeats):
            inner_train_df, valid_df = stratified_split_from_scratch(
                train_df,
                label_col="label",
                test_ratio=valid_ratio,
                random_seed=base_seed + inner_id
            )

            metrics = train_and_evaluate_lr(
                train_df=inner_train_df,
                test_df=valid_df,
                C_value=C_value
            )

            inner_f1_scores.append(metrics["f1"])

        tuning_records.append({
            "C": C_value,
            "mean_validation_f1": np.mean(inner_f1_scores),
            "std_validation_f1": np.std(inner_f1_scores, ddof=1)
        })

    tuning_df = pd.DataFrame(tuning_records)

    best_C = tuning_df.sort_values(
        by="mean_validation_f1",
        ascending=False
    ).iloc[0]["C"]

    return best_C, tuning_df

In [10]:
all_results = []
all_tuning_results = []

for repeat_id in range(1, N_REPEATS + 1):

    print("=" * 60)
    print(f"Outer Repeat {repeat_id:02d}")
    print("=" * 60)

    repeat_folder = os.path.join(
        DATA_PATH,
        f"repeat_{repeat_id:02d}"
    )

    train_path = os.path.join(repeat_folder, "feature_A_train.csv")
    test_path = os.path.join(repeat_folder, "feature_A_test.csv")

    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    print("Train shape:", train_df.shape)
    print("Test shape:", test_df.shape)

    best_C, tuning_df = tune_lr_C_from_scratch(
        train_df=train_df,
        C_values=C_VALUES,
        n_inner_repeats=N_INNER_REPEATS,
        valid_ratio=VALID_RATIO,
        base_seed=1000 + repeat_id * 10
    )

    print("Best C:", best_C)

    tuning_df["outer_repeat"] = repeat_id
    all_tuning_results.append(tuning_df)

    final_metrics = train_and_evaluate_lr(
        train_df=train_df,
        test_df=test_df,
        C_value=best_C
    )

    result_row = {
        "outer_repeat": repeat_id,
        "best_C": best_C,
        **final_metrics
    }

    all_results.append(result_row)

    print("Accuracy :", round(final_metrics["accuracy"], 4))
    print("Precision:", round(final_metrics["precision"], 4))
    print("Recall   :", round(final_metrics["recall"], 4))
    print("F1       :", round(final_metrics["f1"], 4))
    print("AUC      :", round(final_metrics["auc"], 4))

Outer Repeat 01
Train shape: (160000, 17)
Test shape: (39999, 17)
Best C: 10.0
Accuracy : 0.6342
Precision: 0.6364
Recall   : 0.6254
F1       : 0.6309
AUC      : 0.6668
Outer Repeat 02
Train shape: (160000, 17)
Test shape: (39999, 17)
Best C: 0.1
Accuracy : 0.6316
Precision: 0.6322
Recall   : 0.6282
F1       : 0.6302
AUC      : 0.6629
Outer Repeat 03
Train shape: (160000, 17)
Test shape: (39999, 17)
Best C: 0.1
Accuracy : 0.6349
Precision: 0.6348
Recall   : 0.6345
F1       : 0.6347
AUC      : 0.6666
Outer Repeat 04
Train shape: (160000, 17)
Test shape: (39999, 17)
Best C: 0.1
Accuracy : 0.631
Precision: 0.6341
Recall   : 0.6189
F1       : 0.6264
AUC      : 0.665
Outer Repeat 05
Train shape: (160000, 17)
Test shape: (39999, 17)
Best C: 0.1
Accuracy : 0.6331
Precision: 0.6349
Recall   : 0.6254
F1       : 0.6301
AUC      : 0.6653
Outer Repeat 06
Train shape: (160000, 17)
Test shape: (39999, 17)
Best C: 0.1
Accuracy : 0.6395
Precision: 0.6398
Recall   : 0.6377
F1       : 0.6387
AUC      : 

In [11]:
results_df = pd.DataFrame(all_results)
tuning_results_df = pd.concat(all_tuning_results, ignore_index=True)

results_path = os.path.join(OUTPUT_PATH, "LR_FeatureA_repeated_results.csv")
tuning_path = os.path.join(OUTPUT_PATH, "LR_FeatureA_tuning_results.csv")

results_df.to_csv(results_path, index=False, encoding="utf-8-sig")
tuning_results_df.to_csv(tuning_path, index=False, encoding="utf-8-sig")

print("Saved repeated test results to:")
print(results_path)

print("Saved tuning results to:")
print(tuning_path)

results_df

Saved repeated test results to:
LR_FeatureA_Results/LR_FeatureA_repeated_results.csv
Saved tuning results to:
LR_FeatureA_Results/LR_FeatureA_tuning_results.csv


,outer_repeat,best_C,accuracy,precision,recall,f1,tp,tn,fp,fn,auc
0,1,10.0,0.634216,0.636396,0.625413,0.630857,12502,12866,7143,7488,0.666830
1,2,0.1,0.631566,0.632244,0.628164,0.630197,12557,12705,7304,7433,0.662851
2,3,0.1,0.634941,0.634848,0.634467,0.634658,12683,12714,7295,7307,0.666571
3,4,0.1,0.631041,0.634085,0.618859,0.626380,12371,12870,7139,7619,0.665012
4,5,0.1,0.633091,0.634942,0.625413,0.630141,12502,12821,7188,7488,0.665323
5,6,0.1,0.639491,0.639781,0.637669,0.638723,12747,12832,7177,7243,0.670293
6,7,0.1,0.634216,0.634750,0.631416,0.633078,12622,12746,7263,7368,0.667123
7,8,0.1,0.633116,0.633643,0.630315,0.631975,12600,12724,7285,7390,0.666764
8,9,0.1,0.636991,0.638271,0.631566,0.634901,12625,12854,7155,7365,0.667497
9,10,0.1,0.634966,0.635997,0.630365,0.633168,12601,12797,7212,7389,0.669537


In [12]:
summary_records = []

for metric in ["accuracy", "precision", "recall", "f1", "auc"]:
    values = results_df[metric].values

    summary_records.append({
        "metric": metric,
        "mean": np.mean(values),
        "std": np.std(values, ddof=1),
        "standard_error": np.std(values, ddof=1) / np.sqrt(len(values))
    })

summary_df = pd.DataFrame(summary_records)

summary_path = os.path.join(OUTPUT_PATH, "LR_FeatureA_summary.csv")
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

summary_df

,metric,mean,std,standard_error
0,accuracy,0.634363,0.002494,0.000789
1,precision,0.635496,0.002221,0.000702
2,recall,0.629365,0.005260,0.001663
3,f1,0.632408,0.003352,0.001060
4,auc,0.666780,0.002142,0.000677


In [13]:
best_C_frequency = results_df["best_C"].value_counts().reset_index()
best_C_frequency.columns = ["C", "frequency"]

best_C_frequency_path = os.path.join(OUTPUT_PATH, "LR_FeatureA_best_C_frequency.csv")
best_C_frequency.to_csv(best_C_frequency_path, index=False, encoding="utf-8-sig")

best_C_frequency

,C,frequency
0,0.1,9
1,10.0,1
